# Ice Cream Cone Shape Analysis
## Image Analysis - ELC Activity TIET 2029 ECE

In [ ]:
# Import libraries
import cv2
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image

In [ ]:
# Load and preprocess image
image_path = 'softy.jpg'
img = Image.open(image_path)
img_array = np.array(img)

img_rgb = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title('Original Ice Cream Cone')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale')
plt.axis('off')
plt.show()

In [ ]:
# Thresholding and morphological operations
_, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(thresh, cmap='gray')
plt.title('Binary Threshold')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(morph, cmap='gray')
plt.title('Morphological Closing')
plt.axis('off')
plt.show()

In [ ]:
# Edge detection
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 30, 100)

plt.figure(figsize=(8, 6))
plt.imshow(edges, cmap='gray')
plt.title('Edge Detection')
plt.axis('off')
plt.show()

In [ ]:
# Detect cone shape
contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
output_image = img_array.copy()

print(f"Found {len(contours)} contours")

for contour in contours:
    area = cv2.contourArea(contour)
    
    if area > 3000:
        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * perimeter, True)
        
        cv2.drawContours(output_image, [contour], -1, (0, 255, 0), 3)
        
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output_image, (x, y), (x+w, y+h), (255, 0, 0), 2)
        
        aspect_ratio = float(h) / w
        print(f"Ice cream cone detected - Area: {area}, Aspect Ratio: {aspect_ratio:.2f}")
        
        M = cv2.moments(contour)
        if M['m00'] != 0:
            cx = int(M['m10'] / M['m00'])
            cy = int(M['m01'] / M['m00'])
            cv2.putText(output_image, "Ice Cream Cone", (cx-80, cy), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

output_rgb = cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 8))
plt.imshow(output_rgb)
plt.title('Detected Ice Cream Cone')
plt.axis('off')
plt.show()